```python
# Copyright (c) 2026 Louis Gevers
# SPDX-License-Identifier: BSD-3-Clause
```

# Legged Robots Project - Task 2: Flat Terrain Analysis

In this notebook you will analyse the data produced by `01_flat_terrain.ipynb`:
training logs and evaluation rollouts.
There are no simulations here, so it does not need a GPU and can be restarted at any time.

**Before you start:** check out the metrics functions in `src/lr_project/metrics.py`.
This is the only source file that you will have to edit.
We provided two example implementations, but it is up to you to implement the relevant metrics using rollout data (see 2.0c) for your analyses.
There are two kinds of functions, recognizable by their output shape (`N` number of robots, `T` number of timesteps):
- **per-step signals**, shape `(N, T)`: one value per robot and per time step, e.g. to plot a quantity over time.
- **per-robot metrics**, shape `(N,)`: one value per robot, summarizing its whole rollout.

Thanks to `%autoreload`, edits to `metrics.py` are picked up without restarting the kernel.

**Rules for your analysis**
- Every number and figure must come from a cell in this notebook. 
Never copy numbers by hand.
- The sections below list the **minimum** we expect. 
Add any analysis, figure or metric that helps you answer the questions.
- Label every figure: axes with units, a legend when there are several curves, and a title.
- Write your answers under the **[ANSWER]** cells in markdown.
Remember that short and precise beats long.

In [ ]:
%load_ext autoreload
%autoreload 2

import math
import numpy as np
import matplotlib.pyplot as plt

from lr_project.rollout import Rollout
from lr_project import registry, analysis, metrics
from IPython.display import Markdown, display

from lr_project.assets.robots.unitree_go2.go2_constants import get_go2_initial_state

In [ ]:
# One run per step of the priority path: tracking -> energy -> smoothness -> steps -> posture.
# Each run adds reward terms on top of the previous one. Picked by timestamp, never runs[-1].
PHASE_RUNS = {
    "1 tracking": "TODO_YYYY-MM-DD_HH-MM-SS",
    "2 + energy": "TODO_YYYY-MM-DD_HH-MM-SS",
    "3 + smoothness": "TODO_YYYY-MM-DD_HH-MM-SS",
    "4 + foot clearance": "TODO_YYYY-MM-DD_HH-MM-SS",
    "5 + posture (baseline)": "TODO_YYYY-MM-DD_HH-MM-SS",
}
BASELINE = "5 + posture (baseline)"

In [ ]:
# Reward terms each step adds, and the failure that made us add the next ones.
PHASE_CHANGE = {
    "1 tracking": "track_linear_velocity, track_angular_velocity only",
    "2 + energy": "+ joint_torques, joint_power",
    "3 + smoothness": "+ action_rate",
    "4 + foot clearance": "+ foot_clearance, foot_slip",
    "5 + posture (baseline)": "+ body_tilt",
}
PHASE_FAILURE = {
    "1 tracking": "TODO: tracks, but wastes energy",
    "2 + energy": "TODO: efficient, but jittery actions",
    "3 + smoothness": "TODO: smooth, but feet drag",
    "4 + foot clearance": "TODO: clean steps, but body tilts",
    "5 + posture (baseline)": "standing posture (2.1b)",
}

In [ ]:
GRID = "command_grid"     # evaluation shared by every run (notebook 01, eval_name)
JITTER_STRIDE_MULT = 3.5  # action power above 3.5 x stride frequency counts as jitter

In [ ]:
FEET = ["FL", "FR", "RL", "RR"]
JOINTS = [f"{leg}_{joint}" for leg in FEET for joint in ("hip", "thigh", "calf")]  # rollout order
JOINT_COLORS = ["tab:blue", "tab:orange", "tab:green"] * 4                     # hip, thigh, calf
TORQUE_LIMITS = np.array([23.5, 23.5, 45.0] * 4)                                # N m, go2_constants.py

init = get_go2_initial_state().joint_pos
DEFAULT_JOINT_POS = np.array([init[".*_hip_joint"], init[".*_thigh_joint"], init[".*_calf_joint"]] * 4)  # rad

## 2.0 Collected data

We provide you with an example of how to load three kinds of data you have generated:
the list of runs, the training logs of a run, and its evaluation rollouts.
Run it as is, then reuse the patterns in your analysis sections.

### 2.0a Selecting runs of interest

`registry.runs` lists runs filtered by experiment and variant (the names used in `train.train`), sorted by creation time. 
`runs[-1]` is the latest run (it's a python list).

In [ ]:
runs = registry.runs(
    experiment="flat",
    variant="baseline",
)
registry.show(runs)

In [ ]:
run = runs[-1]
registry.show([run])

### 2.0b Training data

`analysis.load_training_logs` reads the curves logged during training and returns a dictionary `{curve name: (iterations, values)}`. 
`patterns` are regular expressions over the curve names, e.g. `"Episode_Reward/.*"` for all reward terms.
The curve names match the ones you see in TensorBoard.

In [ ]:
train_metrics = analysis.load_training_logs(
    run_dir=run.path,
    patterns=["Train/mean_reward", "Train/mean_episode_length", "Episode_Reward/.*"],
)

print("Loaded training metrics:")
for metric in train_metrics.keys():
    print(f"- {metric!r}")

Below we provide an example of plotting training logs, specifically the mean reward per learning iteration:

In [ ]:
iterations, mean_reward = train_metrics["Train/mean_reward"]
print(f"Loaded {len(iterations)} iterations of mean reward with last 3 elements: {mean_reward[-3:]}")

fig, ax = plt.subplots()

ax.plot(iterations, mean_reward)
ax.set_xlabel("Learning iteration")
ax.set_ylabel("Mean reward")

fig.tight_layout()
plt.show()

### 2.0c Evaluation data

`Rollout.load` reads an evaluation saved in notebook 01. 
A `Rollout` holds numpy arrays for N robots and T time steps (see `src/lr_project/rollout.py` for all fields), for example:

| Field | Shape | Content |
| --- | --- | --- |
| `cmd` | (N, 3) | command of each robot $(v_x, v_y, \omega_z)$ |
| `alive` | (N,) | `True` if the robot never fell |
| `base_lin_vel`, `base_ang_vel` | (N, T, 3) | base velocities in the body frame |
| `base_pos`, `base_quat` | (N, T, 3), (N, T, 4) | base position and orientation in the world frame |
| `joint_pos`, `joint_vel`, `joint_torque` | (N, T, 12) | joint states and torques |
| `foot_contact`, `foot_height` | (N, T, 4) | contact flags and heights of the feet (FL, FR, RL, RR) |
| `foot_grf` | (N, T, 4, 3) | ground reaction forces on the feet |

Robots that fell distort most metrics: compute them on `rollouts.select(rollouts.alive)` and report the survival rate separately.

Below we provide some examples of how to take statistics from rollout data and making a simple figure:

In [ ]:
rollouts = Rollout.load(run.eval_file(GRID))

n_robot = len(rollouts.alive)
survive_count = np.sum(rollouts.alive)
survival_rate = np.mean(rollouts.alive)

lin_vel_error = metrics.lin_vel_error(rollouts)

print(f"Survival rate is {survival_rate:.1%} ({np.sum(rollouts.alive)} out of {n_robot} evaluations alive)")
print(f"Average linear velocity error is {np.mean(lin_vel_error):.2f} (std={np.std(lin_vel_error):.2f})")

robot_id = 0
print(f"Robot {robot_id} command: {rollouts.cmd[robot_id]} (lin vel error: {lin_vel_error[robot_id]:.2f})")

fig, axes = plt.subplots(nrows=1, ncols=2)

time = rollouts.dt * np.arange(0, rollouts.base_pos.shape[1])
base_x = rollouts.base_pos[robot_id, :, 0]
base_y = rollouts.base_pos[robot_id, :, 1]
base_z = rollouts.base_pos[robot_id, :, 2]

axes[0].plot(base_x, base_y)
axes[0].set_aspect("equal")
axes[0].set_xlabel("x (m)")
axes[0].set_ylabel("y (m)")

axes[1].plot(time, base_z)
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("z (m)")

fig.suptitle(f"Robot {robot_id} base trajectory")
fig.tight_layout()
plt.show()

## 2.1 Baseline performance

In this section you will characterize your baseline policy: how it learned, how well it performs, and discuss whether it could run on a real robot.

First load the training and rollout data:

In [ ]:
def get_run(created):
    """Run of the flat experiment with this timestamp."""
    for run in registry.runs(experiment="flat"):
        if run.created == created:
            return run
    raise ValueError(f"No run created at {created!r} in ../logs/flat")

In [ ]:
def load_run(created):
    """Training logs and evaluation of one run.
    survivors: robots that never fell, full 10 s (start from rest included, the same for every run)."""
    run = get_run(created)
    train = analysis.load_training_logs(
        run.path, ["Train/mean_reward", "Train/mean_episode_length", "Episode_Reward/.*"])
    rollouts = Rollout.load(run.eval_file(GRID))
    survivors = rollouts.select(rollouts.alive)
    return dict(run=run, train=train, rollouts=rollouts, survivors=survivors)

In [ ]:
data = {phase: load_run(created) for phase, created in PHASE_RUNS.items()}
base = data[BASELINE]
registry.show([d["run"] for d in data.values()])

### 2.1a Training performance

**Minimum:** mean reward and mean episode length curves, and the curves of the individual reward terms.

Questions: 
- When does the policy converge, and would more training help? 
- Which reward terms dominate, and which improve first? 
- Explain how you arrived at your baseline: which settings you tuned, what failures you observed along the way, and how you fixed them.

In [ ]:
# Convergence rule: first iteration after which the 20-iteration moving average
# stays within 2% of the final plateau (mean of the last 50 iterations).
def convergence_iteration(iterations, values, window=20, tol=0.02, tail=50):
    """First iteration after which the `window` moving average stays within `tol`
    of the mean of the last `tail` values. None if it never settles."""
    smooth = np.convolve(values, np.ones(window) / window, mode="valid")
    plateau = np.mean(values[-tail:])
    outside = np.flatnonzero(np.abs(smooth - plateau) > tol * abs(plateau))
    first = 0 if len(outside) == 0 else outside[-1] + 1
    if first >= len(smooth):
        return None
    return iterations[first + window - 1]

In [ ]:
# Figure 1: mean reward and episode length of the baseline, per iteration.
# Dashed line: convergence iteration. Answers whether training converged.
train = base["train"]
panels = {
    "Train/mean_reward": "Mean episode reward",
    "Train/mean_episode_length": "Mean episode length (steps)",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), layout="constrained")
for ax, (tag, label) in zip(axes, panels.items()):
    iterations, values = train[tag]
    ax.plot(iterations, values)
    converged = convergence_iteration(iterations, values)
    if converged is not None:
        ax.axvline(converged, color="k", ls="--", label=f"converged at {converged}")
        ax.legend()
    ax.set(xlabel="Learning iteration", ylabel=label, title=label)
fig.suptitle("Baseline training (dashed: moving average stays within 2% of the final plateau)")
plt.show()

In [ ]:
# Final value and slope over the last 100 iterations of each Figure 1 curve.
# A slope near 0% means more training would not help.
for tag in panels:
    iterations, values = train[tag]
    final = np.mean(values[-50:])
    change_per_100 = 100 * np.polyfit(iterations[-100:], values[-100:], 1)[0]
    print(f"{tag}: final {final:.3f}, last 100 it: {change_per_100:+.3f} per 100 it ({change_per_100 / abs(final):+.1%})")

In [ ]:
# Every Episode_Reward/* curve and its final value (mean of the last 50 iterations).
# Data for Figure 2 and Table A, nothing is plotted here.
terms = {}
for tag, curve in train.items():
    if tag.startswith("Episode_Reward/"):
        terms[tag.removeprefix("Episode_Reward/")] = curve
final = {name: np.mean(values[-50:]) for name, (_, values) in terms.items()}

In [ ]:
# Figure 2: task rewards (left) and penalties (right) per iteration.
# Dotted line: episode length at 90%; earlier changes mostly come from surviving longer.
ep_iterations, ep_length = train["Train/mean_episode_length"]
survival_learned = metrics.iteration_at_90(ep_iterations, ep_length)

fig, (ax_task, ax_penalty) = plt.subplots(1, 2, figsize=(16, 4.5), layout="constrained")
for name, (iterations, values) in terms.items():
    ax = ax_task if final[name] > 0 else ax_penalty
    ax.plot(iterations, values, label=name)
for ax, title in [(ax_task, "Task rewards"), (ax_penalty, "Penalties")]:
    ax.axvline(survival_learned, color="grey", ls=":", label="episode length at 90%")
    ax.set(xlabel="Learning iteration", ylabel="Reward per second", title=title)
    ax.legend(fontsize=8)
plt.show()

In [ ]:
# Renders rows of strings as a Markdown table,
# so Tables A and B stay readable in the saved notebook.
def show_table(header, rows):
    """Markdown table; rows are lists of already formatted strings."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for row in rows:
        lines.append("| " + " | ".join(row) + " |")
    display(Markdown("\n".join(lines)))

In [ ]:
# Table A: final value of each term (mean of the last 50 iterations) and its share of the total |reward|.
# The share shows which terms dominate the policy at the end of training.
total = sum(abs(value) for value in final.values())
rows = []
for name in sorted(terms, key=lambda name: -abs(final[name])):
    rows.append([name, f"{final[name]:+.3f}", f"{abs(final[name]) / total:.1%}"])
show_table(["term", "final (per s)", "share"], rows)

In [ ]:
# Flags robots commanded (0, 0, 0).
# Splits standing robots (standing deviation) from moving ones (CoT, jitter).
def is_standing(r):
    """(N,) True for robots commanded (0, 0, 0)."""
    return np.all(r.cmd == 0, axis=1)

In [ ]:
# Our metrics, one value per robot: standing joint deviation, tilt RMS, jitter fraction.
# Standing joint deviation is our own metric (template) and the 2.2 target.
def standing_joint_deviation(r):
    """(N,) mean |q - q_default| over joints and time (rad). Meaningful for standing commands only."""
    return np.mean(np.abs(r.joint_pos - DEFAULT_JOINT_POS), axis=(1, 2))


def tilt_rms(r):
    """(N,) RMS of the base tilt sqrt(roll² + pitch²) (deg)."""
    roll_pitch = metrics.roll_pitch_per_step(r)
    return np.degrees(np.sqrt(np.mean(np.sum(roll_pitch**2, axis=-1), axis=1)))


def jitter_fraction(r):
    """(N,) fraction of action power above JITTER_STRIDE_MULT x the robot's own stride frequency."""
    return metrics.action_hf_fraction(r, JITTER_STRIDE_MULT * metrics.stride_frequency(r))

In [ ]:
# Table B rows: the 5 runs of the tuning path on the same grid.
# Fall rate on all robots; the rest on survivors (CoT, jitter: moving robots only).
rows = []
for phase, d in data.items():
    r_all, r = d["rollouts"], d["survivors"]
    moving = ~is_standing(r)
    rows.append([
        phase,
        PHASE_CHANGE[phase],
        PHASE_FAILURE[phase],
        f"{1 - r_all.alive.mean():.1%}",
        f"{metrics.lin_vel_error(r).mean():.3f}",
        f"{metrics.yaw_rate_error(r).mean():.3f}",
        f"{np.nanmean(metrics.cost_of_transport(r)[moving]):.3f}",
        f"{np.nanmean(jitter_fraction(r)[moving]):.1%}",
        f"{tilt_rms(r).mean():.2f}",
    ])

In [ ]:
# Table B: one row per step of the tuning path, from the first run to the baseline.
# Separate cell, so the table can be reformatted without recomputing.
show_table(["phase", "change", "failure → next phase", "fall rate", "lin. error (m/s)",
            "yaw error (rad/s)", "CoT", "jitter", "tilt RMS (deg)"], rows)

**[ANSWER]** *Write your answer to the questions here, 2 paragraphs max*

### 2.1b Evaluation performance

**Minimum:**
- velocity tracking: linear and yaw rate errors, overall and per command
- cost of transport
- base stability, with a metric of your own design
- foot ground reaction forces
- a foot contact plot (which foot is on the ground over time)

Questions: 
- Which commands are tracked well, and which are hardest? 
- Which gait did the policy learn, and how can you tell from the contact plot? 
- Is the motion efficient and smooth?

In [ ]:
# Falls of the baseline (all robots). Every later metric uses survivors.
r_all, r = base["rollouts"], base["survivors"]
print(f"Fall rate: {1 - r_all.alive.mean():.1%} ({np.sum(~r_all.alive)} of {r_all.num_robots} robots)")
print("Commands with falls:", np.unique(r_all.cmd[~r_all.alive], axis=0).tolist())

In [ ]:
def command_heatmap(ax, r, values, cmap, title):
    """Mean ± std of a per-robot metric for each command: columns vx, rows (vy, wz). Grey = no survivor."""
    vxs = np.unique(r.cmd[:, 0])
    turns = np.unique(r.cmd[:, 1:], axis=0)
    mean = np.full((len(turns), len(vxs)), np.nan)
    for j, turn in enumerate(turns):
        for k, vx in enumerate(vxs):
            trials = values[(r.cmd[:, 0] == vx) & np.all(r.cmd[:, 1:] == turn, axis=1)]
            if len(trials):
                mean[j, k] = trials.mean()
                ax.text(k, j, f"{trials.mean():.3f}\n± {trials.std():.3f}", ha="center", va="center", fontsize=8)
    cmap = plt.get_cmap(cmap).copy()
    cmap.set_bad("lightgray")
    image = ax.imshow(mean, cmap=cmap, aspect="auto")
    ax.set_xticks(range(len(vxs)), [f"{vx:g}" for vx in vxs])
    ax.set_yticks(range(len(turns)), [f"{vy:g}, {wz:g}" for vy, wz in turns])
    ax.set(title=title, xlabel="vx command (m/s)", ylabel="(vy (m/s), wz (rad/s)) command")
    ax.figure.colorbar(image, ax=ax)

In [ ]:
# Figure 4: RMS tracking error per robot, mean ± std over the robots of each command.
lin_err = metrics.lin_vel_error(r)
yaw_err = metrics.yaw_rate_error(r)

fig, (ax_lin, ax_yaw) = plt.subplots(1, 2, figsize=(18, 6), layout="constrained")
command_heatmap(ax_lin, r, lin_err, "Reds", "Linear velocity RMS error (m/s)")
command_heatmap(ax_yaw, r, yaw_err, "Blues", "Yaw rate RMS error (rad/s)")
plt.show()

In [ ]:
print(f"Linear error {lin_err.mean():.3f} ± {lin_err.std():.3f} m/s")
print(f"Yaw error {yaw_err.mean():.3f} ± {yaw_err.std():.3f} rad/s")

In [ ]:
def per_vx(r, values):
    """Per-robot values grouped by vx command: (sorted vx values, list of arrays)."""
    vxs = np.unique(r.cmd[:, 0])
    return vxs, [values[r.cmd[:, 0] == vx] for vx in vxs]

In [ ]:
# Figure 5: mechanical CoT per vx command (vy and wz pooled), mean ± std over robots.
# vx = 0 is left out: CoT is undefined when standing still.
walking = r.select(r.cmd[:, 0] != 0)
vxs, cot = per_vx(walking, metrics.cost_of_transport(walking))
cot_mean = [np.nanmean(c) for c in cot]
cot_std = [np.nanstd(c) for c in cot]

fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
ax.errorbar(vxs, cot_mean, cot_std, fmt="o-", capsize=4)
ax.set(xlabel="vx command (m/s)", ylabel="Mechanical CoT (-)", title="Cost of transport per forward command")
plt.show()

In [ ]:
# Figure 6a: base height std, roll RMS and pitch RMS per vx command.
stability = metrics.base_stability(r)  # (N, 3): height std (m), roll RMS (rad), pitch RMS (rad)
columns = {
    "Base height std (mm)": 1000 * stability[:, 0],
    "Roll RMS (deg)": np.degrees(stability[:, 1]),
    "Pitch RMS (deg)": np.degrees(stability[:, 2]),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
for ax, (label, values) in zip(axes, columns.items()):
    vxs, by_vx = per_vx(r, values)
    ax.boxplot(by_vx, tick_labels=[f"{vx:g}" for vx in vxs])
    ax.set(xlabel="vx command (m/s)", ylabel=label, title=label)
plt.show()

In [ ]:
# Our metric on the standing robots: mean |q - q_default| over joints and time.
standing = r.select(is_standing(r))
deviation = standing_joint_deviation(standing).mean()
print(f"Standing joint deviation: {deviation:.3f} rad ({np.degrees(deviation):.1f} deg), mean over joints")

In [ ]:
# Robot shown in Figure 7: the first survivor of the fastest straight command.
straight = np.flatnonzero((r_all.cmd[:, 1] == 0) & (r_all.cmd[:, 2] == 0) & r_all.alive)
robot = straight[np.argmax(r_all.cmd[straight, 0])]
robot_cmd = r_all.cmd[robot]
print(f"Robot {robot}, command {robot_cmd.tolist()}")

In [ ]:
# Figure 7: vertical GRF in body weights, 1 to 3 s; shaded = stance.
time = np.arange(r_all.base_pos.shape[1]) * r_all.dt
window = (time >= 1) & (time < 3)
grf_z = -r_all.foot_grf[robot, window, :, 2] / metrics.body_weight(r_all)[robot]  # sensor z is negative: flip to upward
contact = r_all.foot_contact[robot, window]

fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True, sharey=True, layout="constrained")
for foot, ax in enumerate(axes):
    ax.fill_between(time[window], 0, 1, where=contact[:, foot], transform=ax.get_xaxis_transform(),
                    color="0.85", step="mid")
    ax.plot(time[window], grf_z[:, foot])
    ax.set_ylabel(f"{FEET[foot]}\nGRF (BW)")
axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"Vertical GRF and stance (shaded), command {robot_cmd.tolist()}")
plt.show()

In [ ]:
def diagonal_pair_agreement(r):
    """(N,) fraction of steps where FL-RR and FR-RL have the same contact state (1 = perfect trot)."""
    contact = r.foot_contact
    fl_rr = np.mean(contact[..., 0] == contact[..., 3], axis=1)
    fr_rl = np.mean(contact[..., 1] == contact[..., 2], axis=1)
    return (fl_rr + fr_rl) / 2

In [ ]:
# Gait of the moving robots.
moving = r.select(~is_standing(r))
print(f"Duty factor             {np.nanmean(metrics.duty_factor(moving)):.2f}")
print(f"Stride frequency        {np.nanmean(metrics.stride_frequency(moving)):.2f} Hz")
print(f"Diagonal pair agreement {np.mean(diagonal_pair_agreement(moving)):.0%}")
print(f"Peak vertical GRF       {np.nanmean(metrics.peak_vertical_grf(moving)):.2f} BW")
print(f"Weight support          {np.nanmean(metrics.weight_support(moving)):.3f} BW")

In [ ]:
# Figure 8b: jitter fraction per vx command (moving robots).
vxs, jitter = per_vx(moving, 100 * jitter_fraction(moving))

fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
ax.boxplot([j[np.isfinite(j)] for j in jitter], tick_labels=[f"{vx:g}" for vx in vxs])
ax.set(xlabel="vx command (m/s)", ylabel="Action power above cutoff (%)", title="Jitter per forward command")
plt.show()

In [ ]:
# Figure 8c: 99th percentile |torque| (against the effort limits) and |joint velocity| per joint.
torque_p99 = np.percentile(np.abs(r.joint_torque), 99, axis=(0, 1))
velocity_p99 = np.percentile(np.abs(r.joint_vel), 99, axis=(0, 1))

fig, (ax_tau, ax_vel) = plt.subplots(1, 2, figsize=(16, 4.5), layout="constrained")
ax_tau.bar(JOINTS, torque_p99, color=JOINT_COLORS)
ax_tau.plot(JOINTS, TORQUE_LIMITS, "r_", markersize=20, label="effort limit")
ax_tau.set(ylabel="p99 |torque| (N m)", title="Torque per joint")
ax_tau.legend()
ax_vel.bar(JOINTS, velocity_p99, color=JOINT_COLORS)
ax_vel.set(ylabel="p99 |joint velocity| (rad/s)", title="Joint velocity per joint")
for ax in (ax_tau, ax_vel):
    ax.tick_params(axis="x", rotation=90)
plt.show()

In [ ]:
print(f"Highest p99 torque / limit: {np.max(torque_p99 / TORQUE_LIMITS):.0%} ({JOINTS[np.argmax(torque_p99 / TORQUE_LIMITS)]})")
print(f"Max |torque|: {np.abs(r.joint_torque).max():.1f} N m")
print(f"Mean action rate: {metrics.action_rate(r).mean():.3f}")

**[ANSWER]** *Write your answer to the questions here, 2 paragraphs max*

### 2.1c Discussion: deployability

Given your previous observations, would your policy work on the real Go2? 
Discuss:
- **Observations:** can each observation be obtained on the real robot, and how noisy would it be?
- **Actions:** are the commanded motions realistic for the motors (look at torques, joint velocities and action smoothness)?
- **Evaluation on hardware:** which of your metrics could you measure on the real robot, and how?
- **Sim-to-real:** which reward terms or settings did you add to make the policy more transferable, and what is still missing?


**[ANSWER]** *Write your discussion here, 3 paragraphs max*

## 2.2 Controlled experiment

In this section you will describe and report the results of your controlled experiment in section 1.5 of notebook 01 like a short scientific study.

### 2.2a Questions and hypotheses

State your research question and your hypotheses:
- **Null hypothesis $H_0$:** the change has no effect on the metrics you chose.
- **Alternative hypothesis $H_1$:** the change affects the metrics, in a direction you predict, and why.

Name the metrics that will enable to reject the null hypothesis in favor of your alternative hypothesis, answering your research question.

**[ANSWER]** *Write your question, hypotheses, and metrics here.*

### 2.2b Methods

Describe exactly what differs between your baseline and your variant, what was held fixed (training budget, evaluation commands, seeds), how many runs each configuration has, and how you decide whether a difference is real (e.g. larger than the spread between seeds). 
Mention possible confounders.

**[ANSWER]** *Write your methods here. 2 paragraphs max.*

### 2.2c Experimental results

**Minimum:** at least 3 figures comparing baseline and variant, using training curves and/or evaluation metrics. 
Show both configurations in the same figure where possible, with the same axes.

In [ ]:
# TODO after training (notebook 01, 1.5): runs of each configuration, one per seed, same GRID and budget.
SEEDS = {
    "baseline": [PHASE_RUNS[BASELINE], "TODO_seed2", "TODO_seed3"],
    "variant": ["TODO_seed1", "TODO_seed2", "TODO_seed3"],
}
CONFIG_COLORS = {"baseline": "tab:gray", "variant": "tab:red"}

In [ ]:
configs = {name: [load_run(created) for created in runs] for name, runs in SEEDS.items()}
registry.show([seed["run"] for seeds in configs.values() for seed in seeds])

In [ ]:
# Target metric of 2.2a, and the robots it applies to.
TARGET_LABEL = "Standing joint deviation (rad)"


def target_metric(r):
    """(n,) target metric for the robots it applies to."""
    return standing_joint_deviation(r)[is_standing(r)]


def seed_means(seeds, metric):
    """(n_seeds,) mean of a per-robot metric for each seed (survivors)."""
    return np.array([np.nanmean(metric(seed["survivors"])) for seed in seeds])

In [ ]:
# Figure 9: training curves, mean over seeds (line) and min-max (band).
# Only terms defined the same way in both configurations (the total reward is not comparable).
tags = ["Train/mean_episode_length", "Episode_Reward/track_linear_velocity", "Episode_Reward/track_angular_velocity"]

fig, axes = plt.subplots(1, len(tags), figsize=(18, 4.5), layout="constrained")
for ax, tag in zip(axes, tags):
    for name, seeds in configs.items():
        n = min(len(seed["train"][tag][0]) for seed in seeds)
        iterations = seeds[0]["train"][tag][0][:n]
        values = np.array([seed["train"][tag][1][:n] for seed in seeds])
        ax.plot(iterations, values.mean(axis=0), color=CONFIG_COLORS[name], label=f"{name} ({len(seeds)} seeds)")
        ax.fill_between(iterations, values.min(axis=0), values.max(axis=0), color=CONFIG_COLORS[name], alpha=0.2)
    ax.set(xlabel="Learning iteration", ylabel=tag.split("/")[1], title=tag)
    ax.legend()
plt.show()

In [ ]:
# Figure 10: target metric, one point per seed; line = mean over seeds.
target = {name: seed_means(seeds, target_metric) for name, seeds in configs.items()}

fig, ax = plt.subplots(figsize=(6, 4.5), layout="constrained")
for k, (name, values) in enumerate(target.items()):
    ax.scatter(np.full(len(values), k), values, color=CONFIG_COLORS[name], s=60)
    ax.hlines(values.mean(), k - 0.2, k + 0.2, color=CONFIG_COLORS[name])
ax.set_xticks(range(len(target)), list(target))
ax.set(ylabel=TARGET_LABEL, title=f"{TARGET_LABEL}, one point per seed")
plt.show()

In [ ]:
# Is the difference real? It must be larger than the spread between seeds of the same configuration.
difference = target["variant"].mean() - target["baseline"].mean()
spread = max(np.ptp(values) for values in target.values())
print(f"Baseline {target['baseline'].mean():.4f}, variant {target['variant'].mean():.4f}")
print(f"Difference {difference:+.4f}, largest seed spread {spread:.4f}")
print("Larger than the spread" if abs(difference) > spread else "NOT larger than the spread")

In [ ]:
# Figure 11: linear tracking error per vx command, mean over seeds with min-max bars.
fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
for k, (name, seeds) in enumerate(configs.items()):
    curves = []
    for seed in seeds:
        vxs, by_vx = per_vx(seed["survivors"], metrics.lin_vel_error(seed["survivors"]))
        curves.append([errors.mean() for errors in by_vx])
    curves = np.array(curves)
    mean = curves.mean(axis=0)
    bars = [mean - curves.min(axis=0), curves.max(axis=0) - mean]
    ax.errorbar(vxs + 0.03 * (k - 0.5), mean, bars, fmt="o-", capsize=4, color=CONFIG_COLORS[name], label=name)
ax.set(xlabel="vx command (m/s)", ylabel="Linear velocity RMS error (m/s)", title="Tracking per forward command")
ax.legend()
plt.show()

In [ ]:
# Figure 12: side effects on walking (moving robots), one point per seed. The change should not degrade them.
def moving_only(metric):
    return lambda r: metric(r)[~is_standing(r)]


side_effects = {
    "Linear error (m/s)": metrics.lin_vel_error,
    "Yaw error (rad/s)": metrics.yaw_rate_error,
    "CoT (-)": metrics.cost_of_transport,
    "Jitter (-)": jitter_fraction,
}

fig, axes = plt.subplots(1, len(side_effects), figsize=(18, 4), layout="constrained")
for ax, (label, metric) in zip(axes, side_effects.items()):
    for k, (name, seeds) in enumerate(configs.items()):
        values = seed_means(seeds, moving_only(metric))
        ax.scatter(np.full(len(values), k), values, color=CONFIG_COLORS[name], s=60)
        ax.hlines(values.mean(), k - 0.2, k + 0.2, color=CONFIG_COLORS[name])
    ax.set_xticks(range(len(configs)), list(configs))
    ax.set(ylabel=label, title=label)
plt.show()

**[ANSWER]** *Describe what each figure shows here. 2 paragraphs max.*

### 2.2e Discussion

What did you expect, and what did you not expect? 
Do the results support $H_1$ and reject $H_0$, and how conclusive are they (number of runs, size of the difference compared to the variation between runs)? 
Explain the mechanism: how does your change lead to the effect you measured? 
What would you test next?

**[ANSWER]** *Describe your interpretations here. Linking your observations back to the course or the literature is encouraged! 3 paragraphs max.*